In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

#tf.keras.utils.set_random_seed(1032)

2024-05-21 08:54:41.046147: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-21 08:54:41.046264: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-21 08:54:41.116034: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-21 08:54:41.261386: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-21 08:54:42.515668: W tensorflow/compiler/tf2

In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_data = test_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-21 08:54:44.423129: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 08:54:44.655778: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 08:54:44.655857: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 08:54:44.660664: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-21 08:54:44.660709: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS =3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def Model(input_shape):

    model = keras.Sequential([
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Flatten(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(256, activation='relu', activity_regularizer = tf.keras.regularizers.L1(0.007)),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(10, activation='softmax', activity_regularizer = tf.keras.regularizers.L1(0.007))
    ])
    
    #, activity_regularizer = tf.keras.regularizers.L2()
    
    
    return model

Model(input_shape)

In [4]:
model = Model(input_shape)
model.compile(optimizer = keras.optimizers.Adam(learning_rate = 0.00125),
             loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
             metrics=['accuracy'])

In [5]:
earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=75, restore_best_weights=True)


history = model.fit(
train_data,
epochs = 400,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping])

Epoch 1/400


2024-05-21 08:54:55.922438: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-05-21 08:54:56.272228: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-05-21 08:55:00.569784: I external/local_xla/xla/service/service.cc:168] XLA service 0x7ff96baa0090 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-21 08:55:00.569821: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2024-05-21 08:55:00.584512: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1716274500.674849     895 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


74/74 [==============================] - 38s 353ms/step - loss: 2.2490 - accuracy: 0.3142 - val_loss: 3.4066 - val_accuracy: 0.1000
Epoch 2/400
74/74 [==============================] - 13s 168ms/step - loss: 1.7546 - accuracy: 0.4109 - val_loss: 3.4264 - val_accuracy: 0.1000
Epoch 3/400
74/74 [==============================] - 13s 170ms/step - loss: 1.6705 - accuracy: 0.4275 - val_loss: 4.3359 - val_accuracy: 0.1600
Epoch 4/400
74/74 [==============================] - 13s 168ms/step - loss: 1.6210 - accuracy: 0.4540 - val_loss: 3.1538 - val_accuracy: 0.1400
Epoch 5/400
74/74 [==============================] - 13s 168ms/step - loss: 1.5869 - accuracy: 0.4583 - val_loss: 2.1569 - val_accuracy: 0.2600
Epoch 6/400
74/74 [==============================] - 13s 169ms/step - loss: 1.5832 - accuracy: 0.4720 - val_loss: 2.0007 - val_accuracy: 0.2600
Epoch 7/400
74/74 [==============================] - 13s 169ms/step - loss: 1.4635 - accuracy: 0.5045 - val_loss: 1.8232 - val_accuracy: 0.4000
Epoc

KeyboardInterrupt: 

In [12]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def HModel(hp):
    model = keras.Sequential([
        
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
       
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        
        keras.layers.Flatten(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=hp.Int("units", min_value=32, max_value=512, step=32), activation="relu"),
        keras.layers.Dense(10, activation='softmax'),
    ])
    
    learning_rate = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
        metrics=["accuracy"],
    )
    return model

HModel(kt.HyperParameters())

tuner = kt.RandomSearch(
    hypermodel=HModel,
    objective="val_accuracy",
    max_trials=3,
    executions_per_trial=2,
    overwrite=True,
    directory="./ACML_Project",
    project_name="ACML",
)
tuner.search_space_summary()


tuner.search(train_data, epochs=5, validation_data=val_data)

models = tuner.get_best_models(num_models=2)
best_model = models[0]
best_model.summary()

#Retraining the model with the best hyperparamters
#Top 2
best = tuner.get_best_hyperparameters(5)
print("best learning rate:"+str(best[0].get("lr")))
print("best num units:"+str(best[0].get("units")))
#Building model
model = HModel(best[0])

#Value             |Best Value So Far |Hyperparameter
#256               |224               |units
#0.00070032        |0.0003743         |lr


Trial 3 Complete [00h 00m 44s]
val_accuracy: 0.35999999940395355

Best val_accuracy So Far: 0.4099999964237213
Total elapsed time: 00h 02m 16s
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 batch_normalization (Batch  (None, 109, 109, 64)      256       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 109, 109,

In [13]:
earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=75, restore_best_weights=True)

history = model.fit(
train_data,
epochs = 400,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping])

Epoch 1/400
74/74 [==============================] - 6s 47ms/step - loss: 2.5631 - accuracy: 0.2758 - val_loss: 3.5842 - val_accuracy: 0.3000
Epoch 2/400
74/74 [==============================] - 4s 45ms/step - loss: 1.6326 - accuracy: 0.3668 - val_loss: 2.7716 - val_accuracy: 0.3400
Epoch 3/400
74/74 [==============================] - 4s 45ms/step - loss: 1.5080 - accuracy: 0.4301 - val_loss: 3.0402 - val_accuracy: 0.1800
Epoch 4/400
74/74 [==============================] - 4s 45ms/step - loss: 1.4862 - accuracy: 0.4314 - val_loss: 2.5105 - val_accuracy: 0.2400
Epoch 5/400
74/74 [==============================] - 4s 46ms/step - loss: 1.4095 - accuracy: 0.4506 - val_loss: 1.7633 - val_accuracy: 0.3600
Epoch 6/400
74/74 [==============================] - 4s 45ms/step - loss: 1.2877 - accuracy: 0.5199 - val_loss: 3.6820 - val_accuracy: 0.1400
Epoch 7/400
74/74 [==============================] - 4s 44ms/step - loss: 1.2883 - accuracy: 0.5006 - val_loss: 12.3375 - val_accuracy: 0.1200
Epoch

KeyboardInterrupt: 

: 

In [6]:
model.evaluate(val_data, return_dict = True)

2/2 [==============================] - 0s 13ms/step - loss: 0.3031 - accuracy: 0.8800


{'loss': 0.3031112849712372, 'accuracy': 0.8799999952316284}

In [13]:
model.evaluate(test_data, return_dict = True)

2/2 [==============================] - 0s 12ms/step - loss: 0.8314 - accuracy: 0.8600


{'loss': 0.8313798308372498, 'accuracy': 0.8600000143051147}

In [14]:
#model.save(filepath = "model_1.keras")

: 